# Adaptive Metropolis-Hastings Sampling with minibayes

This notebook demonstrates the minibayes Model API and AdaptiveMetropolis sampler with two examples:

1. **Normal-Normal Model**: Sampling from a posterior with analytical solution
2. **Bayesian Linear Regression**: 1D regression with priors on slope, intercept, and noise

The AdaptiveMetropolis sampler automatically tunes proposal covariance during warmup using the Haario et al. (2001) algorithm.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import minibayes as mb
from minibayes import Model, dist, viz
from minibayes.diagnostics import effective_sample_size, summary

---
## Example 1: Normal-Normal Posterior

**Setup:**
- Prior: $\mu \sim N(0, \tau^2)$ with $\tau = 1$
- Likelihood: $y_i \sim N(\mu, \sigma^2)$ with $\sigma = 1$ (known)
- Data: $n$ observations

**Analytical posterior:**
$$\mu | y \sim N(\mu_{post}, \sigma^2_{post})$$

where:
- $\mu_{post} = \frac{n \bar{y} / \sigma^2}{n/\sigma^2 + 1/\tau^2}$
- $\sigma^2_{post} = \frac{1}{n/\sigma^2 + 1/\tau^2}$

In [ ]:
# Generate synthetic data
rng = np.random.default_rng(42)

true_mu = 2.5
sigma = 1.0  # known likelihood std
n = 20
data = rng.normal(true_mu, sigma, size=n)

print(f"True mu = {true_mu}")
print(f"Sample mean = {np.mean(data):.3f}")
print(f"n = {n}")

In [ ]:
# Compute analytical posterior
tau = 1.0  # prior std
y_bar = np.mean(data)

posterior_precision = n / sigma**2 + 1 / tau**2
posterior_var = 1 / posterior_precision
posterior_mean = (n * y_bar / sigma**2) / posterior_precision
posterior_std = np.sqrt(posterior_var)

print(f"Analytical posterior: N({posterior_mean:.3f}, {posterior_std:.3f})")

In [ ]:
# Define minibayes model
priors = {"mu": dist.Normal(loc=0, scale=tau)}


def log_likelihood(params: dict[str, float], obs: np.ndarray) -> float:
    """Normal log-likelihood with known variance."""
    mu = params["mu"]
    return dist.Normal(loc=mu, scale=sigma).obs_logp(obs)


model = Model(priors=priors, log_likelihood=log_likelihood)
print(f"Parameters: {model.param_names}")
print(f"Transforms: {model.transforms}")

In [ ]:
# Run MCMC using mb.sample() with Adaptive MH
result = mb.sample(
    model,
    data=data,
    num_samples=2000,
    num_warmup=500,
    sampler="adaptive_mh",  # Uses AdaptiveMetropolis
    seed=42,
)

# Flatten samples for single-chain use (shape: (1, num_samples) -> (num_samples,))
samples = result.samples["mu"].flatten()
ess = effective_sample_size(result.samples["mu"])

print(f"Acceptance rate: {result.acceptance_rate[0]:.1%}")
print(f"Effective Sample Size: {ess:.1f} (out of 2000)")
print(f"Empirical mean: {np.mean(samples):.3f} (analytical: {posterior_mean:.3f})")
print(f"Empirical std: {np.std(samples):.3f} (analytical: {posterior_std:.3f})")

In [ ]:
# Plot results using minibayes.viz
fig = viz.plot_samples(result, params=["mu"])
plt.show()

fig = viz.plot_density(result, params=["mu"])
# Add analytical posterior for comparison
ax = fig.get_axes()[0]
x_analytical = np.linspace(1.5, 3.0, 200)
analytical_pdf = (
    1 / np.sqrt(2 * np.pi * posterior_var)
) * np.exp(-0.5 * (x_analytical - posterior_mean) ** 2 / posterior_var)
ax.plot(x_analytical, analytical_pdf, color=viz.PALETTE["sage"], lw=2, label="Analytical")
ax.axvline(true_mu, color=viz.PALETTE["terracotta"], linestyle=":", lw=2, label=f"True={true_mu}")
ax.legend()
plt.show()

---
## Example 2: Bayesian Linear Regression

**Model:**
$$y = \alpha + \beta x + \epsilon, \quad \epsilon \sim N(0, \sigma^2)$$

**Priors:**
- $\alpha \sim N(0, 10)$ (intercept)
- $\beta \sim N(0, 10)$ (slope)
- $\sigma \sim \text{HalfNormal}(5)$ (noise std)

With Adaptive MH, we don't need to manually tune proposal scales — the sampler learns appropriate scales and correlations during warmup.

In [ ]:
# Generate synthetic data
rng = np.random.default_rng(123)

true_alpha = 1.5
true_beta = 2.0
true_sigma = 0.3
n = 10

x = np.linspace(0, 3, n)
y = true_alpha + true_beta * x + rng.normal(0, true_sigma, n)

print(f"True parameters: alpha={true_alpha}, beta={true_beta}, sigma={true_sigma}")
print(f"Data: {n} points")

In [ ]:
# Plot data with minibayes style
with viz.style():
    plt.figure(figsize=(8, 5))
    plt.scatter(x, y, alpha=0.7, color=viz.PALETTE["terracotta"], label="Data")
    plt.plot(x, true_alpha + true_beta * x, color=viz.PALETTE["sage"], linestyle="--", lw=2, label="True line")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title("Synthetic Linear Regression Data")
    plt.legend()
    plt.show()

In [ ]:
# Define minibayes model
priors = {
    "alpha": dist.Normal(loc=0, scale=10),
    "beta": dist.Normal(loc=0, scale=10),
    "sigma": dist.HalfNormal(scale=5),
}


def log_likelihood(params: dict[str, float], obs: tuple) -> float:
    """Normal log-likelihood for linear regression."""
    x_data, y_data = obs
    alpha = params["alpha"]
    beta = params["beta"]
    sigma = params["sigma"]

    # Predicted values
    predicted = alpha + beta * x_data

    return dist.Normal(loc=predicted, scale=sigma).obs_logp(y_data)


model = Model(priors=priors, log_likelihood=log_likelihood)
print(f"Parameters: {model.param_names}")
print(f"Transforms: {model.transforms}")

In [ ]:
# Prior predictive check - does our prior cover reasonable data ranges?
x_plot = np.linspace(-1, 4, 100)

def predictive_fn(params, rng):
    """Generate predictions from parameters."""
    mu = params["alpha"] + params["beta"] * x_plot
    return {"y_pred": dist.Normal(mu, params["sigma"]).sample(size=len(x_plot), rng=rng)}

prior_ppc = mb.sample_prior_predictive(model, predictive_fn, num_samples=200, seed=42)

# Plot prior predictive distribution
with viz.style():
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(x, y, alpha=0.9, s=60, label="Data", zorder=3, color=viz.PALETTE["terracotta"])
    for i in range(0, 200, 4):
        ax.plot(x_plot, prior_ppc["y_pred"][i], color=viz.PALETTE["blue"], alpha=0.08, zorder=1)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("Prior Predictive Check: Do our priors cover the data?")
    ax.legend()
    ax.set_ylim(-30, 30)
    plt.show()

print("Prior predictive samples show the range of predictions before seeing data.")
print("If the data falls outside this range, consider adjusting priors.")

In [ ]:
# Run MCMC using mb.sample() - Adaptive MH learns proposal scales automatically!
result = mb.sample(
    model,
    data=(x, y),
    num_samples=5000,
    num_warmup=1000,
    sampler="adaptive_mh",
    seed=42,
)

# Flatten samples for single-chain use (shape: (1, num_samples) -> (num_samples,))
samples = {name: arr.flatten() for name, arr in result.samples.items()}
print(f"Acceptance rate: {result.acceptance_rate[0]:.1%}")
print(f"Elapsed time: {result.elapsed_time:.2f}s")

In [ ]:
# Summary statistics using diagnostics.summary()
samples_arrays = {name: np.array(samples[name]) for name in samples}
stats = summary(samples_arrays)

print("\nPosterior Summary:")
print("-" * 70)
print(f"{'Parameter':<10} {'True':>8} {'Mean':>8} {'Std':>8} {'ESS':>8} {'95% CI'}")
print("-" * 70)

for name, true_val in [
    ("alpha", true_alpha),
    ("beta", true_beta),
    ("sigma", true_sigma),
]:
    s = stats[name]
    print(
        f"{name:<10} {true_val:>8.3f} {s['mean']:>8.3f} {s['std']:>8.3f} {s['ess']:>8.1f}   [{s['5%']:.3f}, {s['95%']:.3f}]"
    )

## Diagnostics

minibayes provides convergence diagnostics to assess MCMC quality:
- **ESS (Effective Sample Size)**: Accounts for autocorrelation in the chain. Higher is better.
- **R-hat**: Requires multiple chains to detect non-convergence (not shown in this single-chain example).

In [ ]:
# Posterior predictive plot using viz.plot_predictive
x_plot = np.linspace(0, 3, 100)

def predictive_mean(params, rng):
    """Mean prediction without observation noise."""
    return {"y_mean": params["alpha"] + params["beta"] * x_plot}

ppc_mean = mb.sample_posterior_predictive(result, predictive_mean, num_samples=500, seed=42)

fig = viz.plot_predictive(x_plot, ppc_mean["y_mean"], x_obs=x, y_obs=y, credible_interval=0.9)
ax = fig.get_axes()[0]
ax.plot(x_plot, true_alpha + true_beta * x_plot, color=viz.PALETTE["sage"], linestyle="--", lw=2, label="True")
ax.legend()
ax.set_title("Posterior Predictive (Mean) - Adaptive MH")
plt.show()

In [ ]:
# Posterior predictive with observation noise (full predictive distribution)
x_plot = np.linspace(-5, 9, 100)

def predictive_with_noise(params, rng):
    """Prediction with observation noise."""
    mu = params["alpha"] + params["beta"] * x_plot
    return {"y_pred": dist.Normal(mu, params["sigma"]).sample(size=len(x_plot), rng=rng)}

ppc = mb.sample_posterior_predictive(result, predictive_with_noise, seed=42)

fig = viz.plot_predictive(x_plot, ppc["y_pred"], x_obs=x, y_obs=y, credible_interval=0.9)
ax = fig.get_axes()[0]
ax.plot(x_plot, true_alpha + true_beta * x_plot, color=viz.PALETTE["sage"], linestyle="--", lw=2, label="True")
ax.set_xlim(-5, 9)
ax.legend()
ax.set_title("90% Predictive Interval (Adaptive MH)")
plt.show()

In [ ]:
# Prediction on NEW DATA - a key use case for predictive sampling
# The predictive function captures new x values via closure

# Define new data points we want to predict (not in training set)
x_new = np.array([4.0, 5.0, 6.0, 7.0])  # Beyond training range [0, 3]

def predict_new_data(params, rng):
    """Predict y for new x values."""
    mu = params["alpha"] + params["beta"] * x_new
    return {
        "y_mean": mu,  # Mean prediction (no noise)
        "y_pred": dist.Normal(mu, params["sigma"]).sample(size=len(x_new), rng=rng),  # With noise
    }

# Generate predictions for new data
ppc_new = mb.sample_posterior_predictive(result, predict_new_data, seed=42)

# Summarize predictions
print("Predictions for new x values:")
print("-" * 60)
print(f"{'x':>6} {'y_true':>10} {'y_mean':>10} {'y_std':>10} {'90% CI'}")
print("-" * 60)

for i, x_val in enumerate(x_new):
    y_true = true_alpha + true_beta * x_val
    y_mean = np.mean(ppc_new["y_pred"][:, i])
    y_std = np.std(ppc_new["y_pred"][:, i])
    y_lo = np.percentile(ppc_new["y_pred"][:, i], 5)
    y_hi = np.percentile(ppc_new["y_pred"][:, i], 95)
    print(f"{x_val:>6.1f} {y_true:>10.2f} {y_mean:>10.2f} {y_std:>10.2f}   [{y_lo:.2f}, {y_hi:.2f}]")

print("\nNote: Uncertainty increases for extrapolation (x > 3)")
print(f"Prediction shape: {ppc_new['y_pred'].shape}  # (num_samples, num_new_points)")

In [ ]:
# Parameter overview using viz
fig = viz.plot_forest(result)
plt.show()

fig = viz.plot_autocorr(result, max_lag=40)
plt.show()

# Summary table
print(viz.summary_table(result))

---
## Comparison: Manual Tuning vs Adaptive

| Aspect | MetropolisHastings | AdaptiveMetropolis |
|--------|-------------------|--------------------|
| **Proposal scales** | Manual per-parameter | Automatic |
| **Correlations** | Independent proposals | Learns correlations |
| **Tuning effort** | Trial and error | None |
| **Warmup** | Optional | Required for adaptation |
| **Memory** | Minimal | Stores samples during warmup |

In the linear regression example above, the Adaptive MH sampler automatically learned:
1. Appropriate scales for each parameter
2. Correlations between parameters (especially α and β)

This makes it much easier to use "out of the box" compared to manually tuning proposal scales.